# Phase 4 — Data Preprocessing

**Goal:** create reproducible preprocessing snapshots while keeping model-time preprocessing leakage-safe.

Snapshots are generated when this notebook runs; no processed datasets are packaged in the ZIP. Phase 8 refits preprocessing inside every walk-forward training fold.

## 1. Load target-integrated data

In [1]:
import sys, gc
from pathlib import Path
PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "project_config.py").is_file(): PROJECT_ROOT = candidate; break
sys.path.insert(0, str(PROJECT_ROOT))
from src.project_config import *
import json
import numpy as np
import pandas as pd
if not TARGET_DATA.exists(): raise FileNotFoundError("Run 03_target_construction.ipynb first.")
df=pd.read_csv(TARGET_DATA, parse_dates=["datetime","target_datetime"])
fit_mask=df["target_datetime"].notna() & (df["target_datetime"]<=pd.Timestamp(TRAIN_TARGET_END))
print("Input rows:",f"{len(df):,}")
print("Audit fit rows:",f"{int(fit_mask.sum()):,}")

Input rows: 420,768
Audit fit rows: 298,356


## 2. Missing-value handling

In [2]:
from sklearn.impute import SimpleImputer
missing_cols=CONTINUOUS+CATEGORICAL
before_missing=df[missing_cols].isna().sum()
num_imputer=SimpleImputer(strategy="median"); cat_imputer=SimpleImputer(strategy="most_frequent")
num_imputer.fit(df.loc[fit_mask,CONTINUOUS]); cat_imputer.fit(df.loc[fit_mask,CATEGORICAL])
stage1=df.copy(); stage1[CONTINUOUS]=num_imputer.transform(stage1[CONTINUOUS]); stage1[CATEGORICAL]=cat_imputer.transform(stage1[CATEGORICAL])
after_missing=stage1[missing_cols].isna().sum(); filled=int(before_missing.sum()-after_missing.sum())
PROCESSED_DIR.mkdir(parents=True,exist_ok=True); stage1.to_csv(CLEAN_STAGE1,index=False)
display(pd.DataFrame({"before":before_missing,"after":after_missing}))
print("Filled predictor/categorical cells:",f"{filled:,}"); print("Target missing remains:",int(stage1[TARGET].isna().sum())); print("CREATED:",CLEAN_STAGE1)
del num_imputer,cat_imputer,before_missing,after_missing; gc.collect()

,before,after
PM2.5,8739,0
PM10,6449,0
SO2,9021,0
NO2,12116,0
CO,20701,0
O3,13277,0
TEMP,398,0
PRES,393,0
DEWP,403,0
WSPM,318,0


Filled predictor/categorical cells: 73,637
Target missing remains: 8751
CREATED: /Users/ruchi/Downloads/phase12_fixed/data/processed/01_after_missing_value_handling.csv


79

## 3. Outlier treatment

In [3]:
outlier_rows=[]; stage2=stage1.copy()
for col in CONTINUOUS:
    s=stage1.loc[fit_mask,col]; valid=s.dropna(); q1=valid.quantile(.25); q3=valid.quantile(.75); iqr=q3-q1
    low=q1-1.5*iqr; high=q3+1.5*iqr
    count=int(((valid<low)|(valid>high)).sum()) if iqr>0 else 0; pct=count/len(valid)*100 if len(valid) else 0.0
    if iqr>0 and pct<=5.0:
        m=(stage2[col]<low)|(stage2[col]>high); capped=int(m.sum()); stage2[col]=stage2[col].clip(low,high); decision="Cap rare IQR extremes (<=5% training outliers)"
    elif iqr==0:
        capped=0; decision="Retain; zero-IQR / zero-inflated"
    else:
        capped=0; decision="Retain; high IQR-flag rate treated as distributional tail"
    outlier_rows.append([col,q1,q3,iqr,low,high,count,pct,capped,decision])
outlier_table=pd.DataFrame(outlier_rows,columns=["variable","Q1","Q3","IQR","lower_fence","upper_fence","train_outlier_count","train_outlier_percent","capped_cells","decision"]).set_index("variable").sort_values("train_outlier_percent",ascending=False)
stage2.to_csv(CLEAN_STAGE2,index=False); display(outlier_table.round(3))
print("Rows deleted: 0"); print("Total capped cells:",int(outlier_table["capped_cells"].sum())); print("Target changed: False"); print("CREATED:",CLEAN_STAGE2)
del stage1; gc.collect()

,Q1,Q3,IQR,lower_fence,upper_fence,train_outlier_count,train_outlier_percent,capped_cells,decision
variable,,,,,,,,,
SO2,3.0,22.0,19.0,-25.50,50.50,26659,8.935,0,Retain; high IQR-flag rate treated as distribu...
CO,500.0,1500.0,1000.0,-1000.00,3000.00,18957,6.354,0,Retain; high IQR-flag rate treated as distribu...
WSPM,0.9,2.1,1.2,-0.90,3.90,17809,5.969,0,Retain; high IQR-flag rate treated as distribu...
PM2.5,23.0,113.0,90.0,-112.00,248.00,13690,4.588,18915,Cap rare IQR extremes (<=5% training outliers)
O3,12.0,82.0,70.0,-93.00,187.00,12549,4.206,16931,Cap rare IQR extremes (<=5% training outliers)
PM10,39.0,150.0,111.0,-127.50,316.50,9490,3.181,13334,Cap rare IQR extremes (<=5% training outliers)
NO2,25.0,71.0,46.0,-44.00,140.00,5872,1.968,7957,Cap rare IQR extremes (<=5% training outliers)
DEWP,-7.4,15.6,23.0,-41.90,50.10,1,0.000,1,Cap rare IQR extremes (<=5% training outliers)
TEMP,4.4,23.5,19.1,-24.25,52.15,0,0.000,0,Cap rare IQR extremes (<=5% training outliers)


Rows deleted: 0
Total capped cells: 57141
Target changed: False
CREATED: /Users/ruchi/Downloads/phase12_fixed/data/processed/02_after_outlier_handling.csv


0

## 4. Categorical encoding audit snapshot

In [4]:
wind_dummies=pd.get_dummies(stage2["wd"],prefix="wd",dtype=np.int8).reindex(columns=[f"wd_{x}" for x in WIND_DIRECTIONS],fill_value=0)
station_dummies=pd.get_dummies(stage2["station"],prefix="station",dtype=np.int8).reindex(columns=[f"station_{x}" for x in EXPECTED_STATIONS],fill_value=0)
stage3=pd.concat([stage2,wind_dummies,station_dummies],axis=1); stage3.to_csv(ENCODE_STAGE3,index=False)
print("Added wind columns:",len(wind_dummies.columns)); print("Added station columns:",len(station_dummies.columns)); print("CREATED:",ENCODE_STAGE3)
del stage3,wind_dummies,station_dummies; gc.collect()

Added wind columns: 16
Added station columns: 12
CREATED: /Users/ruchi/Downloads/phase12_fixed/data/processed/03_after_categorical_encoding.csv


0

## 5. Scaling audit snapshot

In [5]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler(); scaler.fit(stage2.loc[fit_mask,CONTINUOUS])
stage4=stage2.copy(); stage4[CONTINUOUS]=scaler.transform(stage4[CONTINUOUS]); stage4.to_csv(SCALE_STAGE4,index=False)
print("Training mean abs max after scaling:",float(stage4.loc[fit_mask,CONTINUOUS].mean().abs().max())); print("Training std min after scaling:",float(stage4.loc[fit_mask,CONTINUOUS].std(ddof=0).min())); print("Target remains unscaled."); print("CREATED:",SCALE_STAGE4)
del stage4,scaler; gc.collect()

Training mean abs max after scaling: 1.3181082634390264e-14
Training std min after scaling: 0.9999999999999998
Target remains unscaled.
CREATED: /Users/ruchi/Downloads/phase12_fixed/data/processed/04_after_scaling_preview.csv


0

## 6. Supervised audit snapshot and summary

In [6]:
final_cols=[c for c in stage2.columns if c!="No"]; valid=stage2[HAS_NEXT].eq(True)&stage2[TARGET].notna()
stage5_path=PROCESSED_DIR/"05_final_supervised_pre_feature_dataset.csv"
stage2.loc[valid,final_cols].to_csv(stage5_path,index=False)
summary=pd.DataFrame([
["Input / target-integrated",len(df),len(df.columns),"Target constructed; raw predictors intact"],
["01 missing-value handling",len(stage2),len(stage2.columns),f"{filled:,} predictor/categorical cells filled from training statistics"],
["02 outlier handling",len(stage2),len(stage2.columns),f"{int(outlier_table['capped_cells'].sum()):,} cells capped; 0 rows deleted"],
["03 categorical encoding",len(stage2),len(stage2.columns)+28,"28 one-hot columns added in audit snapshot"],
["04 scaling preview",len(stage2),len(stage2.columns),"Continuous predictors standardized using training-only audit parameters"],
["05 supervised pre-feature snapshot",int(valid.sum()),len(final_cols),f"{int((~valid).sum()):,} rows without observed next-hour target excluded"]
],columns=["stage","rows","columns","effect"])
display(summary); summary.to_csv(PROCESSED_DIR/"preprocessing_stage_summary.csv",index=False)
params={"fit_rule":"rows with target_datetime <= 2015-12-31 23:00:00","numeric_missing":"median","categorical_missing":"most_frequent","outlier_rule":"Training-only 1.5*IQR screening; cap rare variables with <=5% training outliers; retain high-tail variables","outlier_exception":["RAIN"],"outlier_screened_columns":CONTINUOUS,"wind_categories":WIND_DIRECTIONS,"station_categories":EXPECTED_STATIONS,"audit_scaling_columns":CONTINUOUS,"target_never_imputed":True,"rows_input":int(len(df)),"rows_supervised_audit":int(valid.sum()),"cells_capped":int(outlier_table["capped_cells"].sum())}
(PROCESSED_DIR/"preprocessing_parameters.json").write_text(json.dumps(params,indent=2),encoding="utf-8")
print("CREATED:",stage5_path); print("Preprocessing audit complete.")

,stage,rows,columns,effect
0,Input / target-integrated,420768,22,Target constructed; raw predictors intact
1,01 missing-value handling,420768,22,"73,637 predictor/categorical cells filled from..."
2,02 outlier handling,420768,22,"57,141 cells capped; 0 rows deleted"
3,03 categorical encoding,420768,50,28 one-hot columns added in audit snapshot
4,04 scaling preview,420768,22,Continuous predictors standardized using train...
5,05 supervised pre-feature snapshot,412017,21,"8,751 rows without observed next-hour target e..."


CREATED: /Users/ruchi/Downloads/phase12_fixed/data/processed/05_final_supervised_pre_feature_dataset.csv
Preprocessing audit complete.
